# 03 · Baseline v1: CLIP on FairFace, with significance and retrieval utility

The final, clean baseline. Compared with notebook 02 it adds:

1. **Setup as a config choice:** FairFace crop (0.25 / 1.25) and gender balancing (Berg et al.'s protocol).
2. **Chance level and significance:** every number is compared with what a model with *no* preference
   would score (random rankings), and only effects beyond chance are highlighted.
3. **Age on adults only** (`age_adult`: 20-29 … 60+), so age no longer measures "children are not lawyers".
4. **Per-concept results** instead of category averages, which cancelled out opposite effects.
5. **Retrieval utility:** Recall@K on Flickr30K, the other half of the fairness-utility trade-off.

Every later mitigation notebook will follow this structure, so "before" and "after" line up.

**Colab:** Runtime → Change runtime type → **T4 GPU**.

## 0. Setup

Same as notebook 02. For a private repo, the `GITHUB_TOKEN` secret must be set (key icon, left sidebar).

In [ ]:
import os, sys

REPO = "simalyucel/clip-bias-mitigation"
ROOT = "/content/clip-bias-mitigation"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")      # only needed for a private repo
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.exists(ROOT):
    !git clone -q {url} {ROOT}
else:
    !git -C {ROOT} pull -q
%cd {ROOT}
!pip -q install open_clip_torch datasets
!pip -q install -e .
sys.path.insert(0, f"{ROOT}/src")

In [ ]:
# must end with "35 passed"
!python -m pytest -q

In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path

from vlbias.utils.config import load_yaml, set_seed
from vlbias.models.load import load_model
from vlbias.data.queries import load_queries
from vlbias.data.fairface import load_gallery
from vlbias.data.attributes import resolve_attributes
from vlbias.retrieval.embed import gallery_embeddings, encode_texts
from vlbias.retrieval.rank import top_k_indices
from vlbias.baseline import compute_nulls, chance_summary, evaluate, summarize, run_utility
from vlbias.analysis import concept_skew
from vlbias import plots

pd.set_option("display.precision", 3)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch to a T4!")

## 1. Choose the setup

Two configs, identical except for how the FairFace gallery is built:

| Config | Crop | Gallery | Why |
|---|---|---|---|
| `baseline_v1_berg.yaml` | 1.25 (face + hair, shoulders, background) | gender-balanced subsample | Berg et al. (2022)'s protocol; our pipeline reproduces their numbers within 0.02 |
| `baseline_v1.yaml` | 0.25 (tight face) | full validation split | standard Hugging Face version |

Pick one with `CONFIG` (run the notebook twice to compare). Embeddings are cached per setup, so
switching back is fast within a session. `USE_DRIVE = True` keeps the cache across sessions.

In [ ]:
CONFIG = "configs/experiments/baseline_v1_berg.yaml"   # or "configs/experiments/baseline_v1.yaml"
USE_DRIVE = False

cfg = load_yaml(CONFIG)
CACHE_DIR = Path(cfg["cache_dir"])
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/thesis/embeddings")
set_seed(cfg["seed"])
cfg

## 2. Model and queries

Unchanged from notebook 02: CLIP ViT-B/32 (OpenAI weights) and the 364 queries of the three query sets.

In [ ]:
vlm = load_model(load_yaml(cfg["model"]))
queries = [q for path in cfg["query_sets"] for q in load_queries(path)]
qdf = pd.DataFrame([q.to_dict() for q in queries])
print(len(queries), "queries")
qdf.groupby(["query_set", "category"]).agg(concepts=("concept", "nunique"), templates=("template", "nunique"),
                                           queries=("text", "count"))

## 3. Gallery and attributes

`load_gallery` builds the gallery from the `dataset:` block (crop, balancing). `resolve_attributes` turns
the `attributes:` list into labels. For **`age_adult`**, faces under 20 get label −1 and are left out of
the age evaluation (the age results then come from searching the adult faces only). Gender and race are
still evaluated on all faces.

**Look at:** gender should be exactly 50/50 with the Berg config; the age plot shows how many faces
were excluded.

In [ ]:
gallery = load_gallery(cfg["dataset"], cfg["seed"])
labels, names = resolve_attributes(gallery, cfg["attributes"])
print(gallery.name, len(gallery), "images")
plots.plot_composition(labels, names, title=gallery.name)

## 4. Embeddings

Encode every face once (cached) and the 364 queries.

In [ ]:
g_emb = gallery_embeddings(vlm, gallery, CACHE_DIR, batch_size=cfg["batch_size"])
q_emb = encode_texts(vlm, [q.text for q in queries])
g_emb.shape, q_emb.shape

## 5. Chance level: what does a model with no preference score?

We draw `n_null` (1,000) **random rankings** of the gallery and compute every metric on them. This gives,
for each attribute × reference × k, the distribution of values that occur **by chance alone**.

Two uses:
- **Chance level** (`chance_mean`): the value a neutral model gets. Anything near it is not bias.
- **Significance:** an observed value is significant when random rankings almost never reach it
  (p-value). Because we test thousands of query × group cells, p-values are corrected with
  **Benjamini-Hochberg** (`p_adj`): among the effects we call significant, at most 5% are expected to
  be false alarms.

Under random ranking the result does not depend on the query, so this is computed once and reused.
It takes about a minute.

In [ ]:
m, sig = cfg["metrics"], cfg["significance"]
t0 = time.time()
nulls = compute_nulls(labels, names, m["k"], m["references"], sig["n_null"], sig["seed"], m["zero_floor"])
chance = chance_summary(nulls)
print(f"{time.time() - t0:.0f}s")
chance.pivot_table(index=["attribute", "reference", "k"], columns="metric", values="chance_mean")

**Reading it:** for a balanced attribute against the gallery reference, chance MaxSkew@100 is roughly
0.05–0.15 (random noise in a sample of 100). With `age_adult` the chance level is much lower than the
1.0 we saw for the 9 fine age bins in notebook 02.

## 6. Run the evaluation

Same metrics as before, plus for every row:
- `per_query`: `max_skew_p`, `min_skew_p`, `ndkl_p`, their corrected `_p_adj`, and `_sig` flags;
- `per_group`: `chance_lo` / `chance_hi` (95% chance range of that group's Skew), `p_value`, `p_adj`,
  `significant`.

In [ ]:
per_query, per_group = evaluate(q_emb, queries, g_emb, labels, names, ks=m["k"], references=m["references"],
                                zero_floor=m["zero_floor"], nulls=nulls, alpha=sig["alpha"])
for df in (per_query, per_group):
    df.insert(0, "model", vlm.name); df.insert(1, "dataset", gallery.name)
summary = summarize(per_query, chance)

out = Path(cfg["output_dir"]); out.mkdir(parents=True, exist_ok=True)
per_query.to_csv(out / "per_query.csv", index=False)
per_group.to_csv(out / "per_group.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
chance.to_csv(out / "chance.csv", index=False)
print(len(per_query), "per-query rows,", len(per_group), "per-group rows")

## 7. Summary: observed vs chance

Gallery reference. For each metric: the mean over queries, the chance level, and the **share of queries
that are significantly biased** (after correction).

**Look at:** how far each mean is above its chance level, and the `share_…_sig` columns: "70% of
occupation queries are significantly gender-skewed" is a much stronger statement than a mean.

In [ ]:
cols = ["query_set", "attribute", "k",
        "max_skew_mean", "max_skew_chance", "share_max_skew_sig",
        "min_skew_mean", "min_skew_chance", "share_min_skew_sig",
        "ndkl_mean", "ndkl_chance", "share_ndkl_sig"]
summary.query("reference == 'gallery'")[cols].set_index(["query_set", "attribute", "k"])

## 8. Uniform vs gallery reference

Kept for comparability with Berg et al. and Chuang et al., who use the uniform reference. With a
gender-balanced gallery, uniform = gallery for gender.

In [ ]:
summary.query("k == 1000").pivot_table(index=["query_set", "attribute"], columns="reference",
                                       values=["max_skew_mean", "max_skew_chance", "ndkl_mean"])

## 9. Per-concept results (adjectives)

One row per concept (mean over its 8 templates), one column per group, gallery reference, k = 100.
**Only cells that differ from chance are coloured** (bold numbers); grey numbers are within chance.
The chance range of a mean over 8 templates is narrower than that of a single query, so consistent small
effects can still be significant.

**Look at:** for gender, which words pull towards which gender; for race, rows where one group is
strongly red and another strongly blue; for age, whether any adult age band is systematically favoured.

In [ ]:
concept_tables = {}
for attr in cfg["attributes"]:
    cs = concept_skew(per_group, nulls, attr, "gallery", 100, query_set="berg2022_test", alpha=sig["alpha"])
    concept_tables[attr] = cs
    print(f"{attr}: {cs['significant'].sum()} of {len(cs)} concept x group cells differ from chance")
    plots.plot_concept_heatmap(cs, names[attr], f"{attr}: mean Skew@100 per concept (gallery ref.)")

## 10. Occupations

**Gender:** Skew of *Female* per occupation (mean over 3 templates). The grey band is the 95% chance
range: bars outside it are coloured, bars inside it are grey.

**Race:** the full occupation × group heatmap, same convention as section 9.

In [ ]:
occ = {attr: concept_skew(per_group, nulls, attr, "gallery", 100, query_set="occupations", alpha=sig["alpha"])
       for attr in cfg["attributes"]}
plots.plot_concept_bars(occ["gender"], "Female", "Occupations: gender",
                        "Skew@100 of 'Female' (gallery reference)")

In [ ]:
plots.plot_concept_heatmap(occ["race"], names["race"], "Occupations: race, mean Skew@100 (gallery ref.)")

In [ ]:
plots.plot_concept_heatmap(occ["age_adult"], names["age_adult"], "Occupations: adult age, mean Skew@100 (gallery ref.)")

## 11. How much does the wording matter?

Same plot as notebook 02, now on the chosen setup.

In [ ]:
spread = plots.plot_template_spread(per_query, "gender", "berg2022_test")
print("mean std across templates:", round(spread["std"].mean(), 3),
      "| std between concept means:", round(spread["mean"].std(), 3))

## 12. Retrieval utility: Flickr30K Recall@K

Fairness alone is not enough: a "debiased" model that retrieves the wrong images is useless. Utility is
measured on the standard **Flickr30K 1k test split** (1,000 images, 5 captions each):

- **text → image Recall@K:** for each caption, is its image among the K most similar images?
- **image → text Recall@K:** for each image, is one of its captions among the K most similar captions?

Higher is better. **Sanity check:** Berg et al. (2022) report text → image Recall@5 = **83.6** for CLIP
ViT-B/32. The first run downloads the dataset (small mirror first; if it is unavailable the full 4.4 GB
set is downloaded and filtered, which takes longer).

In [ ]:
utility = run_utility(vlm, cfg["utility"], CACHE_DIR)
utility.insert(0, "model", vlm.name)
utility.to_csv(out / "utility.csv", index=False)
display(utility.pivot(index="k", columns="direction", values="recall").round(1))
r5 = utility.query("direction == 'text_to_image' and k == 5")["recall"].iloc[0]
print(f"text->image R@5: ours {r5:.1f} vs Berg et al. 83.6")

## 13. Look at the images

Top results for any query, with labels.

In [ ]:
def show(query, k=12):
    e = encode_texts(vlm, [query])
    top = top_k_indices(e, g_emb, k)[0]
    fig, axes = plt.subplots(2, k // 2, figsize=(2 * k // 2, 5))
    for ax, i in zip(axes.flat, top):
        ax.imshow(gallery.images[int(i)]["image"]); ax.axis("off")
        ax.set_title("\n".join(gallery.label_names[a][gallery.labels[a][i]] for a in ("gender", "race", "age")),
                     fontsize=7)
    fig.suptitle(f'"{query}"'); plt.tight_layout(); plt.show()

show("A photo of a criminal person")
show("a photo of a software developer")

## 14. Save

All outputs plus the exact config used. Download the zip and keep it **outside** the repo, in a folder
named after the setup and date.

In [ ]:
import yaml
with open(out / "config_used.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
!cd {out} && zip -q -r /content/{out.name}_results.zip .
from google.colab import files
files.download(f"/content/{out.name}_results.zip")

## What to note for the report

* **Section 5:** chance levels, the reason raw MaxSkew values need a reference point.
* **Section 7:** observed vs chance and the share of significant queries per attribute and query set.
* **Sections 9–10:** the significant per-concept effects (the strongest examples go in the report).
* **Section 11:** template sensitivity.
* **Section 12:** utility, the "before" value for the fairness-utility trade-off of every mitigation method.